In [13]:
import os
import openai
import pandas as pd
import sqlite3
import requests
import json
from datetime import datetime

API_KEY = "sk-proj-"
client = openai.OpenAI(api_key="028fa2e1-fb69-4cca-89aa-1e11ffc4dcc1",  base_url="https://openai.dplit.com/v1")

In [14]:
import random
import numpy as np

# Set seed for reproducible results
random.seed(42)
np.random.seed(42)

# Generate employee data with intentional issues
departments = ['Engineering', 'Marketing', 'Sales', 'HR', 'Finance', 'Operations']
names = ['Alice Johnson', 'Bob Smith', 'Carol Davis', 'David Wilson', 'Eva Brown', 
         'Frank Miller', 'Grace Lee', 'Henry Taylor', 'Ivy Chen', 'Jack Anderson']

data = []
for i in range(100):
    # Intentional issues: some missing values, inconsistent formats
    name = random.choice(names) + f" {i}" if i % 7 != 0 else None  # Missing names
    dept = random.choice(departments) if i % 11 != 0 else "Unknown"  # Unknown departments
    salary = random.randint(40000, 120000) if i % 13 != 0 else None  # Missing salaries
    performance = random.choice(['Excellent', 'Good', 'Average', 'Poor']) if i % 17 != 0 else None
    
    data.append({
        'employee_id': f"EMP{i:03d}",
        'name': name,
        'department': dept,
        'salary': salary,
        'performance': performance,
        'hire_date': f"202{random.randint(0,3)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    })

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('employee_data.csv', index=False)

# Create SQLite database
conn = sqlite3.connect('employee_data.db')
df.to_sql('employees', conn, if_exists='replace', index=False)
conn.close()

print(f"📊 Generated dataset: {len(df)} rows")
print(f"🔍 Data issues: {df.isnull().sum().sum()} missing values")
print("\nFirst 5 rows:")
print(df.head().to_string())

📊 Generated dataset: 100 rows
🔍 Data issues: 29 missing values

First 5 rows:
  employee_id            name   department    salary performance   hire_date
0      EMP000            None      Unknown       NaN        None  2020-01-24
1      EMP001     Eva Brown 1    Marketing   69256.0        Good  2020-11-24
2      EMP002      Ivy Chen 2  Engineering  117397.0        Poor  2020-01-03
3      EMP003  David Wilson 3    Marketing  106237.0   Excellent  2021-12-21
4      EMP004      Ivy Chen 4           HR   68893.0        Poor  2022-01-25


In [17]:
def call_openai(prompt, model="gpt-3.5-turbo", max_tokens=250):
    
    try:   
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0.9
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
few_shot_prompt = """
Analyze employee performance and provide recommendations. Use this format:

Employee: Alice Johnson, Engineering, $75,000, Good performance
Analysis: Solid performer in technical role. Salary is market-competitive.
Recommendation: Consider for mid-level promotion within 6 months.

Employee: Bob Smith, Sales, $45,000, Poor performance  
Analysis: Underperforming in sales role with below-market salary.
Recommendation: Performance improvement plan or role change needed.

Now analyze these employees:
<<<df.head().to_string()>>>
"""

result = call_openai(few_shot_prompt)
print("📋 Few-Shot Analysis:")
print(result)

📋 Few-Shot Analysis:
Employee: John Doe, Marketing, $60,000, Average performance
Analysis: Meeting expectations in marketing role. Salary is slightly above market average.
Recommendation: Provide opportunities for professional development to improve performance.

Employee: Sarah Lee, Finance, $80,000, Excellent performance
Analysis: Exceeding expectations in finance role. Salary is competitive.
Recommendation: Recognize and reward exceptional performance, consider for leadership roles or additional responsibilities.


In [ ]:
def call_openai(prompt, model="gpt-3.5-turbo", max_tokens=250):
    
    try:   
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0.5
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

In [20]:
few_shot_prompt = """
Analyze employee performance and provide recommendations. Use this format:

Employee: Alice Johnson, Engineering, $75,000, Good performance
Analysis: Solid performer in technical role. Salary is market-competitive.
Recommendation: Consider for mid-level promotion within 6 months.

Employee: Bob Smith, Sales, $45,000, Poor performance  
Analysis: Underperforming in sales role with below-market salary.
Recommendation: Performance improvement plan or role change needed.

Now analyze this employee:
Employee: Carol Davis, Marketing, $65,000, Excellent performance
"""

result = call_openai(few_shot_prompt)
print("📋 Few-Shot Analysis:")
print(result)

📋 Few-Shot Analysis:
Analysis: Carol Davis is excelling in her marketing role with a salary that is slightly above market average. She consistently exceeds goals and demonstrates strong leadership skills within the team.
Recommendation: Consider for a promotion to a senior marketing role or provide opportunities for professional development to continue her growth within the company.


In [21]:
cot_prompt = """
Calculate the average salary by department and identify the department with the highest average.

Data:
Engineering: $80,000, $75,000, $90,000, $85,000
Marketing: $60,000, $65,000, $70,000
Sales: $55,000, $50,000, $60,000, $45,000

Let's think step by step:
1. Calculate average for each department
2. Compare the averages
3. Identify the highest
"""

result = call_openai(cot_prompt)
print("🧠 Chain-of-Thought Analysis:")
print(result)

🧠 Chain-of-Thought Analysis:
1. Average salary for Engineering:
($80,000 + $75,000 + $90,000 + $85,000) / 4 = $82,500

Average salary for Marketing:
($60,000 + $65,000 + $70,000) / 3 = $65,000

Average salary for Sales:
($55,000 + $50,000 + $60,000 + $45,000) / 4 = $52,500

2. Comparing the averages:
Engineering: $82,500
Marketing: $65,000
Sales: $52,500

3. Department with the highest average salary: Engineering with an average salary of $82,500.


In [23]:
# Meta-prompting: Generate and then critique
initial_prompt = """
Write a brief analysis of the challenges in managing remote engineering teams.
Keep it under 100 words.
"""

initial_response = call_openai(initial_prompt)
print("📝 Initial Response:")
print(initial_response)
print("\n" + "="*50 + "\n")

# Meta-prompt: Critique and improve
meta_prompt = f"""
Review this analysis and provide an improved version:

Original: {initial_response}

Please:
1. Identify any weaknesses or missing points
2. Suggest improvements
3. Provide a revised version that's more comprehensive and actionable
"""

improved_response = call_openai(meta_prompt)
improved_response

📝 Initial Response:
Managing remote engineering teams presents challenges such as communication barriers, lack of face-to-face interaction, time zone differences, and difficulties in monitoring productivity. Ensuring effective collaboration, maintaining team cohesion, and providing adequate support and resources are key to overcoming these obstacles. Regular check-ins, clear goals, and utilizing technology for virtual meetings and project management can help mitigate these challenges.




'Weaknesses or missing points:\n1. The analysis does not address the importance of building trust and rapport within remote teams.\n2. It does not mention the potential impact of cultural differences on team dynamics.\n3. It lacks specific strategies for addressing each challenge mentioned.\n\nImprovements:\n1. Include the importance of building trust and rapport within remote teams as a key factor in overcoming communication barriers and fostering collaboration.\n2. Address the potential impact of cultural differences on team dynamics and provide strategies for promoting cultural awareness and inclusivity.\n3. Provide specific, actionable strategies for managing time zone differences, monitoring productivity, and maintaining team cohesion.\n\nRevised version:\nManaging remote engineering teams presents unique challenges that require a combination of effective strategies to ensure success. Communication barriers, lack of face-to-face interaction, time zone differences, and difficulties

In [24]:
# Meta-prompting: Generate and then critique
initial_prompt = """
Write a brief analysis of the challenges in managing QA Engineering teams.
Keep it under 100 words.
"""

initial_response = call_openai(initial_prompt)
print("📝 Initial Response:")
print(initial_response)
print("\n" + "="*50 + "\n")

# Meta-prompt: Critique and improve
meta_prompt = f"""
Review this analysis and provide an improved version:

Original: {initial_response}

Please:
1. Identify any weaknesses or missing points
2. Suggest improvements
3. Provide a revised version that's more comprehensive and actionable
"""

improved_response = call_openai(meta_prompt)
improved_response

📝 Initial Response:
Managing QA Engineering teams can be challenging due to the complexity of testing processes, varying skill levels within the team, and the need for continuous learning and adaptation to new technologies. Additionally, balancing the team's workload, setting clear goals and expectations, and ensuring effective communication can also be obstacles in managing QA Engineering teams effectively.




"Weaknesses/Missing Points:\n- The analysis does not mention the importance of leadership and mentorship in managing QA Engineering teams.\n- It does not address the significance of creating a positive team culture and fostering collaboration.\n\nImprovements:\n- Include the importance of leadership and mentorship in guiding and developing QA Engineering teams.\n- Highlight the significance of creating a positive team culture and fostering collaboration to enhance team performance.\n\nRevised Version:\nEffectively managing QA Engineering teams requires a multifaceted approach. In addition to the challenges posed by the complexity of testing processes, varying skill levels, and the need for continuous learning and adaptation to new technologies, it is crucial to emphasize the role of leadership and mentorship in guiding and developing team members. Setting clear goals and expectations, balancing the team's workload, and ensuring effective communication are essential aspects of managing 

In [25]:
# Prompt chaining: Multi-step employee analysis

# Step 1: Data extraction
step1_prompt = f"""
Extract key statistics from this employee data:
{df.describe().to_string()}

Provide: total employees, average salary, departments with most employees.
"""

step1_result = call_openai(step1_prompt)
print("📊 Step 1 - Data Extraction:")
print(step1_result)
print("\n" + "="*50 + "\n")

# Step 2: Analysis based on Step 1 results
step2_prompt = f"""
Based on these statistics: {step1_result}

Identify 3 key insights and potential business recommendations.
Focus on actionable insights for management.
"""

step2_result = call_openai(step2_prompt)
print("💡 Step 2 - Business Insights:")
print(step2_result)
print("\n" + "="*50 + "\n")

# Step 3: Action plan
step3_prompt = f"""
Based on the insights: {step2_result}

Create a 3-point action plan with specific next steps and timelines.
Make it practical for HR and management to implement.
"""

step3_result = call_openai(step3_prompt)
print("🎯 Step 3 - Action Plan:")
print(step3_result)

📊 Step 1 - Data Extraction:
Total employees: 92
Average salary: $84,004.58
Departments with most employees: This information is not provided in the data.


💡 Step 2 - Business Insights:
Key insights:
1. The average salary of $84,004.58 is relatively high, indicating that the company may be paying above-average salaries compared to industry standards. This could be a competitive advantage in attracting and retaining top talent.
2. Without information on departments with the most employees, it is difficult to assess if there are any areas of the company that may be overstaffed or understaffed. Understanding the distribution of employees across departments can help optimize resource allocation and improve overall efficiency.
3. With a total of 92 employees, there may be opportunities for cross-departmental collaboration and knowledge sharing. Encouraging teamwork and communication across different departments can lead to increased innovation and productivity.

Potential business recommend